Construcción de la base de datos

Este cuaderno construye la **base de datos analítica** del proyecto a partir de las
fuentes crudas del buró de crédito:

1. **Carga e integración** de las tablas de operaciones, tarjetas y desempeño.
2. **Cruces** por cliente (`IDENTIFICACION`) y por fecha de corte (`FECHA_CORTE`).
3. **Creación de variables** agregadas, razones y transformaciones.
4. **Persistencia** de la base consolidada en `info.pkl`.


**Conceptos clave**
- **SCE** — Sistema Crediticio Extendido = SBS + SC + SICOM + OTROS
- **SF**  — Sistema Financiero formal = SBS + SC
- Cada variable se agrega por **sistema** (SBS/SC/SICOM/OTROS) y **tipo**
  (OP = operaciones, TC = tarjetas), para los periodos M3, M6, M12, M24 y M36.


### Tratamiento visual: agregación de operaciones/tarjetas a nivel cliente

Un cliente puede tener **varias operaciones de crédito y tarjetas** → varias filas en
las tablas de desempeño. Para tener **una sola fila por cliente** se agregan sus
operaciones mes a mes (M1…M13), con dos reglas según el tipo de variable:

- **Días de morosidad → `max`** : la *peor* mora entre todas sus operaciones ese mes.
- **Saldos (deuda, vencido, castigado…) → `sum`** : lo que el cliente debe *en total*.

```
   varias filas (operaciones/tarjetas)            una fila (cliente)
   ────────────────────────────────────  ──►  ──────────────────────
        Operación 1 ┐
        Operación 2 ├─  groupby(cliente)  ──►   Cliente X
        Tarjeta 1   ┘   max(mora) / sum(saldo)
```

**Antes — varias filas por cliente** (tablas de operaciones y tarjetas)

| Cliente | Producto | Mora M1 | Mora M2 | Mora M3 | Saldo M1 | Saldo M2 | Saldo M3 |
|---|---|--:|--:|--:|--:|--:|--:|
| Cliente X | Operación 1 | 0 | 5 | 30 | 1.000 | 950 | 900 |
| Cliente X | Operación 2 | 0 | 0 | 0 | 500 | 480 | 460 |
| Cliente X | Tarjeta 1 | 0 | 0 | 12 | 300 | 320 | 350 |

**Después — una fila por cliente**

| Cliente | Mora M1 | Mora M2 | Mora M3 | Saldo M1 | Saldo M2 | Saldo M3 |
|---|--:|--:|--:|--:|--:|--:|
| Cliente X | 0 | **5** | **30** | **1.800** | **1.750** | **1.710** |

Verificación del mes M3:
- Mora = `max(30, 0, 12)` = **30** → la peor morosidad del cliente ese mes.
- Saldo = `900 + 460 + 350` = **1.710** → la deuda total del cliente ese mes.



In [54]:
import re
import warnings
import pandas as pd
import numpy as np
import pyreadr  # pip install pyreadr
from pathlib import Path
from pandas.errors import PerformanceWarning

warnings.simplefilter("ignore", category=PerformanceWarning)
import pandas as pd

pd.set_option('display.max_columns', None)   # muestra todas
BDD = Path('data/buro')  # cambia a ruta absoluta si lo necesitas
EV  = 'EstructuraVariables'

## 1. Info al punto de observación 

Filtramos `Muestra == 1`.

In [55]:
info = pd.read_csv('DataInicial_01022024_123459.txt', sep=None, engine='python')
info = info[info['Muestra'] == 1].copy()
info.shape

(79591, 25)

In [56]:
info.head()

,IDENTIFICACION,TIPO_ID,FECHA_CORTE,ESTADO_CIVIL,CARGAS,INSTRUCCION,ANTIG_LABORAL,ANTIG_DOMICILIARIA,RELACION_DEPENDENCIA,PERFIL_CLIENTE,INGRESOS,GASTOS,PROVINCIA,CANTON,PARROQUIA,ACTIVIDAD_ECONOMICA,REFINANCIAMIENTO_REESTRUCTURA,MARCA_AYUDA,DPF,COMUNAL,MESES_GRACIA,RENOVACION,OFICINA,SALDO_PROMEDIO_AHORRO,Muestra
4,244F70904DD738B089FC39D4432D2AB6,C,2022-09-30,Union Libre,1,Secundario,39,15,DEPENDIENTE,DEPENDIENTE,439.01,105.36,SANTA ELENA,LA LIBERTAD,LA LIBERTAD,N000000,NaN,NaN,NaN,NaN,0,NaN,LA LIBERTAD,4.46,1
16,3B9DBDEE3B206ED41B865377DC2C48CE,C,2022-06-30,Soltero (a),1,Secundario,29,0,DEPENDIENTE,DEPENDIENTE,429.00,291.91,NaN,NaN,NaN,N000000,NaN,NaN,NaN,NaN,0,NaN,LA LIBERTAD,38.21,1
17,C7D5591188D15C57CF396833AACADA88,C,2022-03-31,Soltero (a),1,Secundario,18,0,DEPENDIENTE,DEPENDIENTE,471.96,136.34,SANTA ELENA,LA LIBERTAD,LA LIBERTAD,N000000,NaN,NaN,NaN,NaN,0,NaN,LA LIBERTAD,0.00,1
19,491F4236D627EA9A823C9F065FDDA3CB,C,2022-03-31,Soltero (a),0,Secundario,36,0,MICROEMPRESA,MICROEMPRESARIO,295.50,163.27,SANTA ELENA,SANTA ELENA,SANTA ELENA,G471101,NaN,NaN,NaN,NaN,0,NaN,LA LIBERTAD,1.43,1
20,50207615DEE7456136F720BCE282EC79,C,2022-06-30,Soltero (a),0,Secundario,42,0,DEPENDIENTE,DEPENDIENTE,661.89,316.75,SANTA ELENA,SANTA ELENA,COLONCHE,N000000,NaN,NaN,NaN,NaN,0,NaN,LA LIBERTAD,2060.00,1


## 2. Tablas de desempeño Op1 + Tc1

Carga, reordena columnas para que coincidan (`c(1,3,4,2,5:111)`), homologa nombres y apila.

In [57]:
Op1 = pd.read_csv('EstructuraVariables/OperacionesTabla1.txt', sep=None, engine='python')
Tc1 = pd.read_csv('EstructuraVariables/TarjetasTabla1.txt',  sep=None, engine='python')

# Reorden c(1,3,4,2,5:111) en R (1-based) -> índices 0-based en Python
order = [0, 2, 3, 1] + list(range(4, 111))
Op1 = Op1.iloc[:, order]

# Forzar mismos nombres en Tc1 (replica colnames(Tc1) <- colnames(Op1))
Tc1.columns = Op1.columns

des = pd.concat([Op1, Tc1], ignore_index=True)
del Op1, Tc1

des.groupby('IDENTIFICACION_SCORE').size().rename('N').reset_index().head()

,IDENTIFICACION_SCORE,N
0,00004D506A7F3EFA28144779522595F7,3
1,0000C1D3A2BF6B164894B1DBDC8B6013,4
2,0001402504D71EE675A7AFFDCA798720,28
3,0001833F20870A30BFA70DDF85739625,16
4,0002092352432818379164B53973DE7E,23


In [58]:
des[des['IDENTIFICACION_SCORE'] == '00004D506A7F3EFA28144779522595F7'].head()

,FECHA_CORTE_PUNTO_CONTROL,TIPO_IDENTIFICACION,IDENTIFICACION_SCORE,TIPO_PERSONA,TIPO_SISTEMA,NUM_OPERACION_OP,FECHA_CONCESION,SALDO_DEUDA_OP_M1,SALDO_XVENCER_OP_M1,SALDO_VENCIDO_OP_M1,SALDO_NDI_OP_M1,SALDO_CCASTIGADA_OP_M1,SALDO_DJUDICIAL_OP_M1,NUM_DVEN_OP_M1,NUMERO_DIAS_MOROSIDAD_OP_M1,SALDO_DEUDA_OP_M2,SALDO_XVENCER_OP_M2,SALDO_VENCIDO_OP_M2,SALDO_NDI_OP_M2,SALDO_CCASTIGADA_OP_M2,SALDO_DJUDICIAL_OP_M2,NUM_DVEN_OP_M2,NUMERO_DIAS_MOROSIDAD_OP_M2,SALDO_DEUDA_OP_M3,SALDO_XVENCER_OP_M3,SALDO_VENCIDO_OP_M3,SALDO_NDI_OP_M3,SALDO_CCASTIGADA_OP_M3,SALDO_DJUDICIAL_OP_M3,NUM_DVEN_OP_M3,NUMERO_DIAS_MOROSIDAD_OP_M3,SALDO_DEUDA_OP_M4,SALDO_XVENCER_OP_M4,SALDO_VENCIDO_OP_M4,SALDO_NDI_OP_M4,SALDO_CCASTIGADA_OP_M4,SALDO_DJUDICIAL_OP_M4,NUM_DVEN_OP_M4,NUMERO_DIAS_MOROSIDAD_OP_M4,SALDO_DEUDA_OP_M5,SALDO_XVENCER_OP_M5,SALDO_VENCIDO_OP_M5,SALDO_NDI_OP_M5,SALDO_CCASTIGADA_OP_M5,SALDO_DJUDICIAL_OP_M5,NUM_DVEN_OP_M5,NUMERO_DIAS_MOROSIDAD_OP_M5,SALDO_DEUDA_OP_M6,SALDO_XVENCER_OP_M6,SALDO_VENCIDO_OP_M6,SALDO_NDI_OP_M6,SALDO_CCASTIGADA_OP_M6,SALDO_DJUDICIAL_OP_M6,NUM_DVEN_OP_M6,NUMERO_DIAS_MOROSIDAD_OP_M6,SALDO_DEUDA_OP_M7,SALDO_XVENCER_OP_M7,SALDO_VENCIDO_OP_M7,SALDO_NDI_OP_M7,SALDO_CCASTIGADA_OP_M7,SALDO_DJUDICIAL_OP_M7,NUM_DVEN_OP_M7,NUMERO_DIAS_MOROSIDAD_OP_M7,SALDO_DEUDA_OP_M8,SALDO_XVENCER_OP_M8,SALDO_VENCIDO_OP_M8,SALDO_NDI_OP_M8,SALDO_CCASTIGADA_OP_M8,SALDO_DJUDICIAL_OP_M8,NUM_DVEN_OP_M8,NUMERO_DIAS_MOROSIDAD_OP_M8,SALDO_DEUDA_OP_M9,SALDO_XVENCER_OP_M9,SALDO_VENCIDO_OP_M9,SALDO_NDI_OP_M9,SALDO_CCASTIGADA_OP_M9,SALDO_DJUDICIAL_OP_M9,NUM_DVEN_OP_M9,NUMERO_DIAS_MOROSIDAD_OP_M9,SALDO_DEUDA_OP_M10,SALDO_XVENCER_OP_M10,SALDO_VENCIDO_OP_M10,SALDO_NDI_OP_M10,SALDO_CCASTIGADA_OP_M10,SALDO_DJUDICIAL_OP_M10,NUM_DVEN_OP_M10,NUMERO_DIAS_MOROSIDAD_OP_M10,SALDO_DEUDA_OP_M11,SALDO_XVENCER_OP_M11,SALDO_VENCIDO_OP_M11,SALDO_NDI_OP_M11,SALDO_CCASTIGADA_OP_M11,SALDO_DJUDICIAL_OP_M11,NUM_DVEN_OP_M11,NUMERO_DIAS_MOROSIDAD_OP_M11,SALDO_DEUDA_OP_M12,SALDO_XVENCER_OP_M12,SALDO_VENCIDO_OP_M12,SALDO_NDI_OP_M12,SALDO_CCASTIGADA_OP_M12,SALDO_DJUDICIAL_OP_M12,NUM_DVEN_OP_M12,NUMERO_DIAS_MOROSIDAD_OP_M12,SALDO_DEUDA_OP_M13,SALDO_XVENCER_OP_M13,SALDO_VENCIDO_OP_M13,SALDO_NDI_OP_M13,SALDO_CCASTIGADA_OP_M13,SALDO_DJUDICIAL_OP_M13,NUM_DVEN_OP_M13,NUMERO_DIAS_MOROSIDAD_OP_M13
375167,2021-12-31,C,00004D506A7F3EFA28144779522595F7,NATURAL,COOPERATIVA,0040191008,2011-03-21,5166.97,0.0,1.0,0.0,5165.97,0.0,0,3477,5166.97,0.0,1.0,0.0,5165.97,0.0,0,3508,5166.97,0.0,1.0,0.0,5165.97,0.0,0,3536,5166.97,0.0,1.0,0.0,5165.97,0.0,0,3567,5166.97,0.0,1.0,0.0,5165.97,0.0,0,3597,5166.97,0.0,1.0,0.0,5165.97,0.0,0,3628,5166.97,0.0,1.0,0.0,5165.97,0.0,0,3658,5166.97,0.0,1.0,0.0,5165.97,0.0,0,3689,5166.97,0.0,1.0,0.0,5165.97,0.0,0,3720,5166.97,0.0,1.0,0.0,5165.97,0.0,0,3750,5166.97,0.0,1.0,0.0,5165.97,0.0,0,3781,5166.97,0.0,1.0,0.0,5165.97,0.0,0,3811,5166.97,0.0,1.0,0.0,5165.97,0.0,0,3842
413646,2023-06-30,C,00004D506A7F3EFA28144779522595F7,NATURAL,COOPERATIVA,0040191008,2011-03-21,5166.97,0.0,1.0,0.0,5165.97,0.0,0,4023,5166.97,0.0,1.0,0.0,5165.97,0.0,0,4054,5166.97,0.0,1.0,0.0,5165.97,0.0,0,4085,5166.97,0.0,1.0,0.0,5165.97,0.0,0,4115,5166.97,0.0,1.0,0.0,5165.97,0.0,0,4146,5166.97,0.0,1.0,0.0,5165.97,0.0,0,4176,5166.97,0.0,1.0,0.0,5165.97,0.0,0,4207,0.00,0.0,0.0,0.0,0.00,0.0,0,0,0.00,0.0,0.0,0.0,0.00,0.0,0,0,0.00,0.0,0.0,0.0,0.00,0.0,0,0,0.00,0.0,0.0,0.0,0.00,0.0,0,0,0.00,0.0,0.0,0.0,0.00,0.0,0,0,0.00,0.0,0.0,0.0,0.00,0.0,0,0
456169,2023-09-30,C,00004D506A7F3EFA28144779522595F7,NATURAL,COOPERATIVA,0040191008,2011-03-21,5166.97,0.0,1.0,0.0,5165.97,0.0,0,4115,5166.97,0.0,1.0,0.0,5165.97,0.0,0,4146,5166.97,0.0,1.0,0.0,5165.97,0.0,0,4176,5166.97,0.0,1.0,0.0,5165.97,0.0,0,4207,0.00,0.0,0.0,0.0,0.00,0.0,0,0,0.00,0.0,0.0,0.0,0.00,0.0,0,0,0.00,0.0,0.0,0.0,0.00,0.0,0,0,0.00,0.0,0.0,0.0,0.00,0.0,0,0,0.00,0.0,0.0,0.0,0.00,0.0,0,0,0.00,0.0,0.0,0.0,0.00,0.0,0,0,0.00,0.0,0.0,0.0,0.00,0.0,0,0,0.00,0.0,0.0,0.0,0.00,0.0,0,0,0.00,0.0,0.0,0.0,0.00,0.0,0,0


## 3. Agregaciones s1-s5 por `IDENTIFICACION_SCORE`

- `s1`: **max** de días de morosidad por mes (M1-M13)
- `s2-s5`: **sum** de saldos por mes

In [59]:
key = ['IDENTIFICACION_SCORE', 'FECHA_CORTE_PUNTO_CONTROL']
meses = range(1, 14)

def agg(prefix, func):
    cols = [f'{prefix}{i}' for i in meses]
    return des.groupby(key)[cols].agg(func).reset_index()

s1 = agg('NUMERO_DIAS_MOROSIDAD_OP_M', 'max')
s2 = agg('SALDO_DEUDA_OP_M',           'sum')
s3 = agg('SALDO_VENCIDO_OP_M',         'sum')
s4 = agg('SALDO_CCASTIGADA_OP_M',      'sum')
s5 = agg('SALDO_DJUDICIAL_OP_M',       'sum')

res = s1.merge(s2, on=key, how='outer')
res = res.merge(s3, on=key, how='outer')
res = res.merge(s4, on=key, how='outer')
res = res.merge(s5, on=key, how='outer')
del s1, s2, s3, s4, s5
res.shape


(261656, 67)

## 4. Cruce desempeño + punto de observación 

In [60]:
res = res.rename(columns={'IDENTIFICACION_SCORE': 'IDENTIFICACION',
                          'FECHA_CORTE_PUNTO_CONTROL': 'FECHA_CORTE'})

# alinear tipos de fecha en ambos lados
res['FECHA_CORTE']  = pd.to_datetime(res['FECHA_CORTE'])
info['FECHA_CORTE'] = pd.to_datetime(info['FECHA_CORTE'])

info = info.merge(res, on=['IDENTIFICACION', 'FECHA_CORTE'], how='left')
del res
info.shape


(79591, 90)

## 5. Tablas históricas Op3 + Tc3

Cruce por **doble llave**: `IDENTIFICACION` y `FECHA_CORTE`.

In [ ]:
Op3 = pd.read_csv('EstructuraVariables/OperacionesTabla3.txt', sep=None, engine='python')
Tc3 = pd.read_csv('EstructuraVariables/TarjetaTabla3.txt',     sep=None, engine='python')

cli = 'F81329F437E438FC89A5F705BC1D50D5'
print(info.loc[info['IDENTIFICACION'] == cli, 'FECHA_CORTE'].head())
print(Op3.loc[Op3['IDENTIFICACION_SCORE'] == cli, 'FECHA_CORTE_PUNTO_CONTROL'].head())

76502   2022-09-30
Name: FECHA_CORTE, dtype: datetime64[us]
0        2022-09-30
9507     2023-06-30
18367    2023-09-30
Name: FECHA_CORTE_PUNTO_CONTROL, dtype: str


In [62]:
# colnames(Op3)[c(3,1)] <- c("IDENTIFICACION", "FECHA_CORTE")  (1-based)
def renombrar(df):
    cols = list(df.columns)
    cols[0] = 'FECHA_CORTE'
    cols[2] = 'IDENTIFICACION'
    df.columns = cols
    return df

Op3 = renombrar(Op3)
Tc3 = renombrar(Tc3)

# Tipo fecha en ambos lados antes del merge
info['FECHA_CORTE'] = pd.to_datetime(info['FECHA_CORTE'])
Op3['FECHA_CORTE']  = pd.to_datetime(Op3['FECHA_CORTE'])
Tc3['FECHA_CORTE']  = pd.to_datetime(Tc3['FECHA_CORTE'])

info = info.merge(Op3, on=['IDENTIFICACION', 'FECHA_CORTE'], how='left')
info = info.merge(Tc3, on=['IDENTIFICACION', 'FECHA_CORTE'], how='left')
del Op3, Tc3
info.shape

(79591, 794)

## 6. Extracción de nombres base SF + SCE 

Normaliza `_OTROS_SIS_` → `_OTROS_` y obtiene la lista única de raíces de variables `_SBS_`.

In [63]:
# Renombre global en info
info.columns = [c.replace('_OTROS_SIS_', '_OTROS_') for c in info.columns]

# Tomamos columnas que contienen _SBS_ y extraemos la raíz quitando _SBS_OP... y _SBS_TC...
cols_sbs   = [c for c in info.columns if '_SBS_' in c]
namesinfo1 = sorted({re.sub(r'_SBS_(OP|TC).*$', '', c) for c in cols_sbs})
print(f'{len(namesinfo1)} raíces únicas')
namesinfo1[:10]

15 raíces únicas


['DEUDA_TOTAL',
 'FECHA_VENC',
 'MAX_DVEN',
 'MVALVEN',
 'MVAL_CASTIGO',
 'MVAL_DEMANDA',
 'NENT_VEN',
 'NOPE_APERT',
 'NTC_APERT',
 'PROM_CAS']

## 7. Base indicadores `d` desde `INDICADORES.RData`

In [64]:
rdata = pyreadr.read_r(str('INDICADORES.RData'))
d = rdata['d']
del rdata
d.columns.tolist()[:20]

['numeroIdentificacion',
 'fechaCalificacion',
 'fechaCorte',
 'tipoIdentificacionSujeto',
 'tipoIdentificacionSujetoDescripcion',
 'identificacionSujeto',
 'score001',
 'segScore002',
 'salOpDiaBan003',
 'salOpDiaCoo004',
 'salOpDiaCom005',
 'salOpDiaSer006',
 'salOpDiaCob007',
 'salOpDia008',
 'salOpVenBan009',
 'salOpVenCoo010',
 'salOpVenCom011',
 'salOpVenSer012',
 'salOpVenCob013',
 'salOpVen014']

In [65]:
# Lista de variables a DESCARTAR (líneas 84-91 R)
vars_drop = [
    'fechaCalificacion', 'tipoIdentificacionSujeto',
    'tipoIdentificacionSujetoDescripcion', 'identificacionSujeto',
    'marcaPrinTC090', 'emisorPrinTC091', 'gastoPersonal093',
    'disponibleEst094', 'entidad109', 'salEntidad110', 'entidad111',
    'salEntidad112', 'entidad113', 'salEntidad114', 'entidad115',
    'salEntidad116', 'buenoMaloBancos342', 'buenoMaloCoops343',
    'buenoMaloTarjetas344', 'Covid19OpTc345', 'ID', 'entidad',
    'ID4', 'ingreso136_Actual',
]

# d <- setDT(d)[, -vars, with=FALSE]
d = d.drop(columns=[c for c in vars_drop if c in d.columns])

# colnames(d)[1:2] <- c("IDENTIFICACION", "FECHA_CORTE")
d.columns = ['IDENTIFICACION', 'FECHA_CORTE'] + list(d.columns[2:])

# d[, FECHA_CORTE := lubridate::ymd(FECHA_CORTE)]
d['FECHA_CORTE'] = pd.to_datetime(d['FECHA_CORTE'])

# info <- d[info, on=c("IDENTIFICACION","FECHA_CORTE")]
info = info.merge(d, on=['IDENTIFICACION', 'FECHA_CORTE'], how='left')
del d
info.shape

(79591, 1115)

## Verificación final

`info` debe tener la misma forma que el `info` de R al final de la línea 100.

In [66]:
print('Filas:', len(info))
print('Columnas:', info.shape[1])
info.head()

Filas: 79591
Columnas: 1115


,IDENTIFICACION,TIPO_ID,FECHA_CORTE,ESTADO_CIVIL,CARGAS,INSTRUCCION,ANTIG_LABORAL,ANTIG_DOMICILIARIA,RELACION_DEPENDENCIA,PERFIL_CLIENTE,INGRESOS,GASTOS,PROVINCIA,CANTON,PARROQUIA,ACTIVIDAD_ECONOMICA,REFINANCIAMIENTO_REESTRUCTURA,MARCA_AYUDA,DPF,COMUNAL,MESES_GRACIA,RENOVACION,OFICINA,SALDO_PROMEDIO_AHORRO,Muestra,NUMERO_DIAS_MOROSIDAD_OP_M1,NUMERO_DIAS_MOROSIDAD_OP_M2,NUMERO_DIAS_MOROSIDAD_OP_M3,NUMERO_DIAS_MOROSIDAD_OP_M4,NUMERO_DIAS_MOROSIDAD_OP_M5,NUMERO_DIAS_MOROSIDAD_OP_M6,NUMERO_DIAS_MOROSIDAD_OP_M7,NUMERO_DIAS_MOROSIDAD_OP_M8,NUMERO_DIAS_MOROSIDAD_OP_M9,NUMERO_DIAS_MOROSIDAD_OP_M10,NUMERO_DIAS_MOROSIDAD_OP_M11,NUMERO_DIAS_MOROSIDAD_OP_M12,NUMERO_DIAS_MOROSIDAD_OP_M13,SALDO_DEUDA_OP_M1,SALDO_DEUDA_OP_M2,SALDO_DEUDA_OP_M3,SALDO_DEUDA_OP_M4,SALDO_DEUDA_OP_M5,SALDO_DEUDA_OP_M6,SALDO_DEUDA_OP_M7,SALDO_DEUDA_OP_M8,SALDO_DEUDA_OP_M9,SALDO_DEUDA_OP_M10,SALDO_DEUDA_OP_M11,SALDO_DEUDA_OP_M12,SALDO_DEUDA_OP_M13,SALDO_VENCIDO_OP_M1,SALDO_VENCIDO_OP_M2,SALDO_VENCIDO_OP_M3,SALDO_VENCIDO_OP_M4,SALDO_VENCIDO_OP_M5,SALDO_VENCIDO_OP_M6,SALDO_VENCIDO_OP_M7,SALDO_VENCIDO_OP_M8,SALDO_VENCIDO_OP_M9,SALDO_VENCIDO_OP_M10,SALDO_VENCIDO_OP_M11,SALDO_VENCIDO_OP_M12,SALDO_VENCIDO_OP_M13,SALDO_CCASTIGADA_OP_M1,SALDO_CCASTIGADA_OP_M2,SALDO_CCASTIGADA_OP_M3,SALDO_CCASTIGADA_OP_M4,SALDO_CCASTIGADA_OP_M5,SALDO_CCASTIGADA_OP_M6,SALDO_CCASTIGADA_OP_M7,SALDO_CCASTIGADA_OP_M8,SALDO_CCASTIGADA_OP_M9,SALDO_CCASTIGADA_OP_M10,SALDO_CCASTIGADA_OP_M11,SALDO_CCASTIGADA_OP_M12,SALDO_CCASTIGADA_OP_M13,SALDO_DJUDICIAL_OP_M1,SALDO_DJUDICIAL_OP_M2,SALDO_DJUDICIAL_OP_M3,SALDO_DJUDICIAL_OP_M4,SALDO_DJUDICIAL_OP_M5,SALDO_DJUDICIAL_OP_M6,SALDO_DJUDICIAL_OP_M7,SALDO_DJUDICIAL_OP_M8,SALDO_DJUDICIAL_OP_M9,SALDO_DJUDICIAL_OP_M10,SALDO_DJUDICIAL_OP_M11,SALDO_DJUDICIAL_OP_M12,SALDO_DJUDICIAL_OP_M13,TIPO_IDENTIFICACION_x,TIPO_PERSONA_x,NOPE_REFIN_OP_3M,NOPE_REFIN_OP_6M,NOPE_REFIN_OP_12M,NOPE_REFIN_OP_24M,FECHA_REFIN_OP_12M,VAL_REFIN_OP_12M,NOPE_XVEN_OP_3M,NOPE_XVEN_OP_6M,NOPE_XVEN_OP_12M,NOPE_XVEN_OP_24M,NOPE_XVEN_OP_36M,NOPE_VENC_OP_3M,NOPE_VENC_OP_6M,NOPE_VENC_OP_12M,NOPE_VENC_OP_24M,NOPE_VENC_OP_36M,NOPE_NDI_OP_3M,NOPE_NDI_OP_6M,NOPE_NDI_OP_12M,NOPE_NDI_OP_24M,NOPE_NDI_OP_36M,NOPE_VENC_1A30_OP_3M,NOPE_VENC_1A30_OP_6M,NOPE_VENC_1A30_OP_12M,NOPE_VENC_1A30_OP_24M,NOPE_VENC_1A30_OP_36M,NOPE_VENC_31A90_OP_3M,NOPE_VENC_31A90_OP_6M,NOPE_VENC_31A90_OP_12M,NOPE_VENC_31A90_OP_24M,NOPE_VENC_31A90_OP_36M,NOPE_VENC_91A180_OP_3M,NOPE_VENC_91A180_OP_6M,NOPE_VENC_91A180_OP_12M,NOPE_VENC_91A180_OP_24M,NOPE_VENC_91A180_OP_36M,NOPE_VENC_181A360_OP_3M,NOPE_VENC_181A360_OP_6M,NOPE_VENC_181A360_OP_12M,NOPE_VENC_181A360_OP_24M,NOPE_VENC_181A360_OP_36M,NOPE_VENC_MAYOR360_OP_3M,NOPE_VENC_MAYOR360_OP_6M,NOPE_VENC_MAYOR360_OP_12M,NOPE_VENC_MAYOR360_OP_24M,NOPE_VENC_MAYOR360_OP_36M,NOPE_DEMANDA_OP_3M,NOPE_DEMANDA_OP_6M,NOPE_DEMANDA_OP_12M,NOPE_DEMANDA_OP_24M,NOPE_DEMANDA_OP_36M,NOPE_CASTIGO_OP_3M,NOPE_CASTIGO_OP_6M,NOPE_CASTIGO_OP_12M,NOPE_CASTIGO_OP_24M,NOPE_CASTIGO_OP_36M,NOPE_APERT_SBS_OP_3M,NOPE_APERT_SBS_OP_6M,NOPE_APERT_SBS_OP_12M,NOPE_APERT_SBS_OP_24M,NOPE_APERT_SBS_OP_36M,NOPE_APERT_SC_OP_3M,NOPE_APERT_SC_OP_6M,NOPE_APERT_SC_OP_12M,NOPE_APERT_SC_OP_24M,NOPE_APERT_SC_OP_36M,NOPE_APERT_SICOM_OP_3M,NOPE_APERT_SICOM_OP_6M,NOPE_APERT_SICOM_OP_12M,NOPE_APERT_SICOM_OP_24M,NOPE_APERT_SICOM_OP_36M,NOPE_APERT_OTROS_OP_3M,NOPE_APERT_OTROS_OP_6M,NOPE_APERT_OTROS_OP_12M,NOPE_APERT_OTROS_OP_24M,NOPE_APERT_OTROS_OP_36M,MVALVEN_SBS_OP_3M,MVALVEN_SBS_OP_6M,MVALVEN_SBS_OP_12M,MVALVEN_SBS_OP_24M,MVALVEN_SBS_OP_36M,MVALVEN_SC_OP_3M,MVALVEN_SC_OP_6M,MVALVEN_SC_OP_12M,MVALVEN_SC_OP_24M,MVALVEN_SC_OP_36M,MVALVEN_SICOM_OP_3M,MVALVEN_SICOM_OP_6M,MVALVEN_SICOM_OP_12M,MVALVEN_SICOM_OP_24M,MVALVEN_SICOM_OP_36M,MVALVEN_OTROS_OP_3M,MVALVEN_OTROS_OP_6M,MVALVEN_OTROS_OP_12M,MVALVEN_OTROS_OP_24M,MVALVEN_OTROS_OP_36M,MVAL_DEMANDA_SBS_OP_3M,MVAL_DEMANDA_SBS_OP_6M,MVAL_DEMANDA_SBS_OP_12M,MVAL_DEMANDA_SBS_OP_24M,MVAL_DEMANDA_SBS_OP_36M,MVAL_DEMANDA_SC_OP_3M,MVAL_DEMANDA_SC_OP_6M,MVAL_DEMANDA_SC_OP_12M,MVAL_DEMANDA_SC_OP_24M,MVAL_D

In [67]:
info.columns.tolist()

['IDENTIFICACION',
 'TIPO_ID',
 'FECHA_CORTE',
 'ESTADO_CIVIL',
 'CARGAS',
 'INSTRUCCION',
 'ANTIG_LABORAL',
 'ANTIG_DOMICILIARIA',
 'RELACION_DEPENDENCIA',
 'PERFIL_CLIENTE',
 'INGRESOS',
 'GASTOS',
 'PROVINCIA',
 'CANTON',
 'PARROQUIA',
 'ACTIVIDAD_ECONOMICA',
 'REFINANCIAMIENTO_REESTRUCTURA',
 'MARCA_AYUDA',
 'DPF',
 'COMUNAL',
 'MESES_GRACIA',
 'RENOVACION',
 'OFICINA',
 'SALDO_PROMEDIO_AHORRO',
 'Muestra',
 'NUMERO_DIAS_MOROSIDAD_OP_M1',
 'NUMERO_DIAS_MOROSIDAD_OP_M2',
 'NUMERO_DIAS_MOROSIDAD_OP_M3',
 'NUMERO_DIAS_MOROSIDAD_OP_M4',
 'NUMERO_DIAS_MOROSIDAD_OP_M5',
 'NUMERO_DIAS_MOROSIDAD_OP_M6',
 'NUMERO_DIAS_MOROSIDAD_OP_M7',
 'NUMERO_DIAS_MOROSIDAD_OP_M8',
 'NUMERO_DIAS_MOROSIDAD_OP_M9',
 'NUMERO_DIAS_MOROSIDAD_OP_M10',
 'NUMERO_DIAS_MOROSIDAD_OP_M11',
 'NUMERO_DIAS_MOROSIDAD_OP_M12',
 'NUMERO_DIAS_MOROSIDAD_OP_M13',
 'SALDO_DEUDA_OP_M1',
 'SALDO_DEUDA_OP_M2',
 'SALDO_DEUDA_OP_M3',
 'SALDO_DEUDA_OP_M4',
 'SALDO_DEUDA_OP_M5',
 'SALDO_DEUDA_OP_M6',
 'SALDO_DEUDA_OP_M7',
 'SALDO_DEU

## Homologación de columnas duplicadas

El renombre `_OTROS_SIS_` → `_OTROS_` puede colisionar con columnas `_OTROS_` ya
existentes y generar **nombres de columna duplicados**. Eso rompe las agregaciones
posteriores, porque `info[col]` devolvería dos columnas en lugar de una. Se conserva
la **primera aparición** de cada nombre.

In [68]:
# %% Eliminar columnas con nombre duplicado (conserva la primera aparición)
dups = sorted(set(info.columns[info.columns.duplicated()]))
print(f'Columnas con nombre duplicado: {len(dups)}')
if dups:
    print('Ejemplos:', dups[:10])

info = info.loc[:, ~info.columns.duplicated()].copy()
print('info tras homologar:', info.shape)

Columnas con nombre duplicado: 10
Ejemplos: ['PROM_MAX_DVEN_OTROS_OP_12M', 'PROM_MAX_DVEN_OTROS_OP_24M', 'PROM_MAX_DVEN_OTROS_OP_36M', 'PROM_MAX_DVEN_OTROS_OP_3M', 'PROM_MAX_DVEN_OTROS_OP_6M', 'PROM_MAX_DVEN_OTROS_TC_12M', 'PROM_MAX_DVEN_OTROS_TC_24M', 'PROM_MAX_DVEN_OTROS_TC_36M', 'PROM_MAX_DVEN_OTROS_TC_3M', 'PROM_MAX_DVEN_OTROS_TC_6M']
info tras homologar: (79591, 1105)


## 8. Creación de variables agregadas SCE y SF

**Definiciones:**
- **SCE** = `SBS + SC + SICOM + OTROS` (sistema crediticio  Ecuatoriano)
- **SF**  = `SBS + SC` (sistema financiero )

Cada bloque combina las columnas por sistema (SBS/SC/SICOM/OTROS) y por tipo (OP=operaciones, TC=tarjetas) usando `sum` o `max` según la naturaleza de la variable, para los periodos M3, M6, M12, M24 (y M36 cuando aplica).

In [69]:
# --- Helpers reutilizables ---------------------------------------------------

def aggregate(info, base, periods, agg, scope='SCE', side=None):
    """Crea info[f'{base}_{scope}[_side]_{p}M'] como agregación entre sistemas.

    scope='SCE' usa SBS+SC+SICOM+OTROS; scope='SF' usa SBS+SC.
    side=None combina OP y TC; 'OP' o 'TC' restringe a un solo tipo.
    agg='sum' o 'max'. Columnas faltantes se ignoran silenciosamente.
    """
    systems = ['SBS', 'SC', 'SICOM', 'OTROS'] if scope == 'SCE' else ['SBS', 'SC']
    sides   = [side] if side else ['OP', 'TC']
    suffix  = scope if side is None else f'{scope}_{side}'
    for p in periods:
        cols = [f'{base}_{s}_{t}_{p}M' for s in systems for t in sides
                if f'{base}_{s}_{t}_{p}M' in info.columns]
        if not cols:
            continue
        out = f'{base}_{suffix}_{p}M'
        info[out] = info[cols].sum(axis=1) if agg == 'sum' else info[cols].max(axis=1)


def system_optc_sum(info, base, system, periods):
    """info[f'{base}_{system}_{p}M'] = OP + TC para ese sistema."""
    for p in periods:
        a, b = f'{base}_{system}_OP_{p}M', f'{base}_{system}_TC_{p}M'
        if a in info.columns and b in info.columns:
            info[f'{base}_{system}_{p}M'] = info[a] + info[b]


def safe_ratio(info, num, den, out):
    """out = num/den si den>0, sino 0 (réplica de ifelse(den>0, num/den, 0))."""
    d = info[den]
    info[out] = np.where(d > 0, info[num] / d.where(d > 0, 1), 0)


def safe_log(info, col, out=None):
    """out = log(col) si col>1, sino 0."""
    x = info[col]
    info[out or f'LN_{col}'] = np.where(x > 1, np.log(x.where(x > 1, 1)), 0)

### 8.1 DEUDA_TOTAL  (sum)

In [70]:
periods_4 = [3, 6, 12, 24]

aggregate(info, 'DEUDA_TOTAL', periods_4, 'sum', scope='SCE')
aggregate(info, 'DEUDA_TOTAL', periods_4, 'sum', scope='SF')


### 8.2 MVALVEN  (max) y agregados por sistema (OP+TC)

In [71]:
periods_5 = [3, 6, 12, 24, 36]

aggregate(info, 'MVALVEN', periods_5, 'max', scope='SCE')
aggregate(info, 'MVALVEN', periods_5, 'max', scope='SF')

for sys in ['SBS', 'SC', 'SICOM', 'OTROS']:
    system_optc_sum(info, 'MVALVEN', sys, periods_5)


### 8.3 MVAL_CASTIGO y MVAL_DEMANDA  (max)

In [72]:
for base in ['MVAL_CASTIGO', 'MVAL_DEMANDA']:
    aggregate(info, base, periods_5, 'max', scope='SCE')
    aggregate(info, base, periods_5, 'max', scope='SF')


### 8.4 NENT_VEN  (sum)

In [73]:
aggregate(info, 'NENT_VEN', periods_5, 'sum', scope='SCE')
aggregate(info, 'NENT_VEN', periods_5, 'sum', scope='SF')


### 8.5 NTC_APERT  (suma SCE solo TC)

In [74]:
# Solo TC, todos los sistemas: NTC_APERT_SCE_TC_pM = SBS_TC + SC_TC + OTROS_TC + SICOM_TC
for p in periods_5:
    cols = [f'NTC_APERT_{s}_TC_{p}M' for s in ['SBS', 'SC', 'OTROS', 'SICOM']
            if f'NTC_APERT_{s}_TC_{p}M' in info.columns]
    if cols:
        info[f'NTC_APERT_SCE_TC_{p}M'] = info[cols].sum(axis=1)


### 8.6 PROM_CAS, PROM_DEM, PROM_NDI, PROM_VEN, PROM_XVEN  (sum)

Y PROM_MAX_DVEN  (max)

In [75]:
for base in ['PROM_CAS', 'PROM_DEM', 'PROM_NDI', 'PROM_VEN', 'PROM_XVEN']:
    aggregate(info, base, periods_5, 'sum', scope='SCE')
    aggregate(info, base, periods_5, 'sum', scope='SF')

aggregate(info, 'PROM_MAX_DVEN', periods_5, 'max', scope='SCE')
aggregate(info, 'PROM_MAX_DVEN', periods_5, 'max', scope='SF')


### 8.7 MAX_DVEN  (max) y agregados por sistema (OP+TC)

In [76]:
aggregate(info, 'MAX_DVEN', periods_5, 'max', scope='SCE')
aggregate(info, 'MAX_DVEN', periods_5, 'max', scope='SF')

for sys in ['OTROS', 'SBS', 'SC', 'SICOM']:
    system_optc_sum(info, 'MAX_DVEN', sys, periods_5)


### 8.8 NOPE_APERT  (sum, sólo OP)

In [77]:
# SCE incluye SBS, SC, OTROS, SICOM (todos OP)
for p in periods_5:
    cols_sce = [f'NOPE_APERT_{s}_OP_{p}M' for s in ['SBS', 'SC', 'OTROS', 'SICOM']
                if f'NOPE_APERT_{s}_OP_{p}M' in info.columns]
    if cols_sce:
        info[f'NOPE_APERT_SCE_OP_{p}M'] = info[cols_sce].sum(axis=1)

    cols_sf = [f'NOPE_APERT_{s}_OP_{p}M' for s in ['SBS', 'SC']
               if f'NOPE_APERT_{s}_OP_{p}M' in info.columns]
    if cols_sf:
        info[f'NOPE_APERT_SF_OP_{p}M'] = info[cols_sf].sum(axis=1)


## 9. Razones DEUDA_TOTAL: cross-period y cross-system

In [78]:
# Razones inter-temporales SCE
safe_ratio(info, 'DEUDA_TOTAL_SCE_6M',  'DEUDA_TOTAL_SCE_12M', 'r_DEUDA_TOTAL_SCE_6A12')
safe_ratio(info, 'DEUDA_TOTAL_SCE_3M',  'DEUDA_TOTAL_SCE_6M',  'r_DEUDA_TOTAL_SCE_3A6')
safe_ratio(info, 'DEUDA_TOTAL_SCE_12M', 'DEUDA_TOTAL_SCE_24M', 'r_DEUDA_TOTAL_SCE_12A24')

# Razones inter-temporales SF
safe_ratio(info, 'DEUDA_TOTAL_SF_6M',  'DEUDA_TOTAL_SF_12M', 'r_DEUDA_TOTAL_SF_6A12')
safe_ratio(info, 'DEUDA_TOTAL_SF_3M',  'DEUDA_TOTAL_SF_6M',  'r_DEUDA_TOTAL_SF_3A6')
safe_ratio(info, 'DEUDA_TOTAL_SF_12M', 'DEUDA_TOTAL_SF_24M', 'r_DEUDA_TOTAL_SF_12A24')

# Razón SF / SCE por periodo
for p in periods_4:
    safe_ratio(info, f'DEUDA_TOTAL_SF_{p}M', f'DEUDA_TOTAL_SCE_{p}M',
               f'r_DEUDA_TOTAL_SFsSCE_{p}M')


In [79]:
# Sumas DEUDA por sistema (OP + TC)
for sys in ['SBS', 'SC', 'SICOM', 'OTROS']:
    system_optc_sum(info, 'DEUDA_TOTAL', sys, periods_4)

# Razones inter-temporales solo para SC
for num, den, out in [
    (3, 6,  'r_DEUDA_TOTAL_SC_3s6M'),
    (3, 12, 'r_DEUDA_TOTAL_SC_3s12M'),
    (3, 24, 'r_DEUDA_TOTAL_SC_3s24M'),
    (6, 12, 'r_DEUDA_TOTAL_SC_6s12M'),
    (6, 24, 'r_DEUDA_TOTAL_SC_6s24M'),
    (12,24, 'r_DEUDA_TOTAL_SC_12s24M'),
]:
    safe_ratio(info, f'DEUDA_TOTAL_SC_{num}M', f'DEUDA_TOTAL_SC_{den}M', out)

# Razones <sistema>/SCE por periodo
for sys in ['SBS', 'SC', 'OTROS', 'SICOM']:
    for p in periods_4:
        safe_ratio(info, f'DEUDA_TOTAL_{sys}_{p}M', f'DEUDA_TOTAL_SCE_{p}M',
                   f'r_DEUDA_TOTAL_{sys}sSCE_{p}M')


## 10. Logaritmo natural de DEUDA_TOTAL por sistema (sólo OP)  

In [80]:
for sys in ['SBS', 'SC', 'OTROS', 'SICOM']:
    for p in periods_4:
        col = f'DEUDA_TOTAL_{sys}_OP_{p}M'
        if col in info.columns:
            safe_log(info, col)


## 11. Razones NOPE_APERT por sistema vs SCE

In [81]:
for sys in ['SBS', 'SC', 'SICOM', 'OTROS']:
    for p in periods_5:
        num = f'NOPE_APERT_{sys}_OP_{p}M'
        den = f'NOPE_APERT_SCE_OP_{p}M'
        if num in info.columns and den in info.columns:
            safe_ratio(info, num, den, f'r_NOPE_APERT_{sys}_OP_{p}MsSCE_{p}M')


## 12. SCE y SF separados por OP y por TC 

Mismas agregaciones pero **sin combinar** OP con TC: se generan columnas tipo `*_SCE_OP_*` y `*_SCE_TC_*`.

In [82]:
# DEUDA_TOTAL: sum, periods 3,6,12,24
for side in ['OP', 'TC']:
    aggregate(info, 'DEUDA_TOTAL', periods_4, 'sum', scope='SCE', side=side)
    aggregate(info, 'DEUDA_TOTAL', periods_4, 'sum', scope='SF',  side=side)

# Variables max con periods 3,6,12,24,36
for base in ['MVALVEN', 'MVAL_CASTIGO', 'MVAL_DEMANDA', 'PROM_MAX_DVEN', 'MAX_DVEN']:
    for side in ['OP', 'TC']:
        aggregate(info, base, periods_5, 'max', scope='SCE', side=side)
        aggregate(info, base, periods_5, 'max', scope='SF',  side=side)

# Variables sum con periods 3,6,12,24,36
for base in ['NENT_VEN', 'PROM_CAS', 'PROM_DEM', 'PROM_NDI', 'PROM_VEN', 'PROM_XVEN']:
    for side in ['OP', 'TC']:
        aggregate(info, base, periods_5, 'sum', scope='SCE', side=side)
        aggregate(info, base, periods_5, 'sum', scope='SF',  side=side)

# NOPE_APERT: solo OP (replica líneas 996-1001)
for p in periods_5:
    cols = [f'NOPE_APERT_{s}_OP_{p}M' for s in ['SBS', 'SC', 'OTROS', 'SICOM']
            if f'NOPE_APERT_{s}_OP_{p}M' in info.columns]
    if cols:
        info[f'NOPE_APERT_SCE_OP_{p}M'] = info[cols].sum(axis=1)


## 13. Razones cross-period y cross-system para OP/TC separados 

In [83]:
# Razones inter-temporales para SCE y SF, separados OP y TC
for side in ['OP', 'TC']:
    for num, den, label in [(6, 12, '6A12'), (3, 6, '3A6'), (12, 24, '12A24')]:
        # SCE
        n = f'DEUDA_TOTAL_SCE_{side}_{num}M'
        d = f'DEUDA_TOTAL_SCE_{side}_{den}M'
        if n in info.columns and d in info.columns:
            safe_ratio(info, n, d, f'r_DEUDA_TOTAL_SCE_{side}_{label}')
        # SF
        n = f'DEUDA_TOTAL_SF_{side}_{num}M'
        d = f'DEUDA_TOTAL_SF_{side}_{den}M'
        if n in info.columns and d in info.columns:
            safe_ratio(info, n, d, f'r_DEUDA_TOTAL_SF_{side}_{label}')

# SF/SCE por periodo, separado OP/TC
for side in ['OP', 'TC']:
    for p in periods_4:
        safe_ratio(info, f'DEUDA_TOTAL_SF_{side}_{p}M',
                         f'DEUDA_TOTAL_SCE_{side}_{p}M',
                         f'r_DEUDA_TOTAL_SFsSCE_{side}_{p}M')

# <sistema>/SCE por periodo y lado
for sys in ['SBS', 'SC', 'OTROS', 'SICOM']:
    for side in ['OP', 'TC']:
        for p in periods_4:
            num = f'DEUDA_TOTAL_{sys}_{side}_{p}M'
            den = f'DEUDA_TOTAL_SCE_{side}_{p}M'
            if num in info.columns and den in info.columns:
                safe_ratio(info, num, den, f'r_DEUDA_TOTAL_{sys}sSCE_{side}_{p}M')


## 14. LN de variables PROM y razones de NOPE basadas en intervalos 

In [84]:
# 14.1 LN de todas las variables PROM_*
variables_PROM = [c for c in info.columns if c.startswith('PROM_')]
for var in variables_PROM:
    safe_log(info, var)

# 14.2 Razones NOPE_<tipo>_<intervalo1>M / NOPE_<tipo>_<intervalo2>M para intervalos consecutivos
variables_NOPE = [c for c in info.columns if c.startswith('NOPE_')]
intervalos = [3, 6, 12, 24, 36]

for i in range(1, len(intervalos)):
    int1, int2 = intervalos[i - 1], intervalos[i]
    for var in variables_NOPE:
        m = re.match(r'^NOPE_(.*?)_\d+M$', var)
        if not m:
            continue
        tipo = m.group(1)
        v1, v2 = f'NOPE_{tipo}_{int1}M', f'NOPE_{tipo}_{int2}M'
        if v1 in info.columns and v2 in info.columns:
            out = f'r_NOPE_{tipo}_{int1}M_s_{tipo}_{int2}M'
            info[out] = np.where(info[v2] == 0, 0, info[v1] / info[v2])


## 15. Limpieza de columnas mal formateadas

In [85]:
# porcentajeUsoTC122: '12,3%' -> 0.123
if 'porcentajeUsoTC122' in info.columns:
    info['porcentajeUsoTC122'] = (
        info['porcentajeUsoTC122'].astype(str)
            .str.replace('%', '', regex=False)
            .str.replace(',', '.', regex=False)
            .replace({'nan': np.nan, 'None': np.nan})
            .astype(float) / 100
    )

# numAcreedoresOPyTC386: limpia '%' y ',', a numérico, NaN -> 1
if 'numAcreedoresOPyTC386' in info.columns:
    info['numAcreedoresOPyTC386'] = (
        info['numAcreedoresOPyTC386'].astype(str)
            .str.replace('%', '', regex=False)
            .str.replace(',', '.', regex=False)
            .replace({'nan': np.nan, 'None': np.nan})
            .astype(float)
    )
    # Réplica del comportamiento de reemplazo_col(info, c('numAcreedoresOPyTC386'), 1)
    info['numAcreedoresOPyTC386'] = info['numAcreedoresOPyTC386'].fillna(1)


## 16. Razones temporales para variables PROM_

In [86]:
variables_PROM = [c for c in info.columns if c.startswith('PROM_')]
intervalos = [3, 6, 12, 24, 36]

for i in range(1, len(intervalos)):
    int1, int2 = intervalos[i - 1], intervalos[i]
    for var in variables_PROM:
        m = re.match(r'^PROM_(.*?)_\d+M$', var)
        if not m:
            continue
        tipo = m.group(1)
        v1, v2 = f'PROM_{tipo}_{int1}M', f'PROM_{tipo}_{int2}M'
        # Validar AMBAS columnas (la versión previa solo verificaba v2 -> KeyError en v1)
        if v1 in info.columns and v2 in info.columns:
            out = f'r_PROM_{tipo}_{int1}s{int2}M'
            # División segura: reemplazo 0 por NaN antes de dividir, luego NaN -> 0
            denom = info[v2].replace(0, np.nan)
            info[out] = (info[v1] / denom).fillna(0)


## 17. ANTIGUEDAD_SCE y ANTIGUEDAD_SF 

In [87]:
ant_cols = [c for c in info.columns if c.startswith('ANTIGUEDAD_')]
print('Columnas ANTIGUEDAD_ detectadas:', ant_cols)

sce_cols = [
    'ANTIGUEDAD_TC_OTROS', 'ANTIGUEDAD_TC_SBS', 'ANTIGUEDAD_TC_SC', 'ANTIGUEDAD_TC_SICOM',
    'ANTIGUEDAD_OP_SBS',   'ANTIGUEDAD_OP_SC',  'ANTIGUEDAD_OP_SICOM', 'ANTIGUEDAD_OP_OTROS',
]
sf_cols = ['ANTIGUEDAD_TC_SBS', 'ANTIGUEDAD_TC_SC', 'ANTIGUEDAD_OP_SBS', 'ANTIGUEDAD_OP_SC']

info['ANTIGUEDAD_SCE'] = info[[c for c in sce_cols if c in info.columns]].sum(axis=1)
info['ANTIGUEDAD_SF']  = info[[c for c in sf_cols  if c in info.columns]].sum(axis=1)


Columnas ANTIGUEDAD_ detectadas: ['ANTIGUEDAD_OP_SBS', 'ANTIGUEDAD_OP_SC', 'ANTIGUEDAD_OP_SICOM', 'ANTIGUEDAD_OP_OTROS', 'ANTIGUEDAD_TC_OTROS', 'ANTIGUEDAD_TC_SBS', 'ANTIGUEDAD_TC_SC', 'ANTIGUEDAD_TC_SICOM']


In [88]:
info.columns.tolist()

['IDENTIFICACION',
 'TIPO_ID',
 'FECHA_CORTE',
 'ESTADO_CIVIL',
 'CARGAS',
 'INSTRUCCION',
 'ANTIG_LABORAL',
 'ANTIG_DOMICILIARIA',
 'RELACION_DEPENDENCIA',
 'PERFIL_CLIENTE',
 'INGRESOS',
 'GASTOS',
 'PROVINCIA',
 'CANTON',
 'PARROQUIA',
 'ACTIVIDAD_ECONOMICA',
 'REFINANCIAMIENTO_REESTRUCTURA',
 'MARCA_AYUDA',
 'DPF',
 'COMUNAL',
 'MESES_GRACIA',
 'RENOVACION',
 'OFICINA',
 'SALDO_PROMEDIO_AHORRO',
 'Muestra',
 'NUMERO_DIAS_MOROSIDAD_OP_M1',
 'NUMERO_DIAS_MOROSIDAD_OP_M2',
 'NUMERO_DIAS_MOROSIDAD_OP_M3',
 'NUMERO_DIAS_MOROSIDAD_OP_M4',
 'NUMERO_DIAS_MOROSIDAD_OP_M5',
 'NUMERO_DIAS_MOROSIDAD_OP_M6',
 'NUMERO_DIAS_MOROSIDAD_OP_M7',
 'NUMERO_DIAS_MOROSIDAD_OP_M8',
 'NUMERO_DIAS_MOROSIDAD_OP_M9',
 'NUMERO_DIAS_MOROSIDAD_OP_M10',
 'NUMERO_DIAS_MOROSIDAD_OP_M11',
 'NUMERO_DIAS_MOROSIDAD_OP_M12',
 'NUMERO_DIAS_MOROSIDAD_OP_M13',
 'SALDO_DEUDA_OP_M1',
 'SALDO_DEUDA_OP_M2',
 'SALDO_DEUDA_OP_M3',
 'SALDO_DEUDA_OP_M4',
 'SALDO_DEUDA_OP_M5',
 'SALDO_DEUDA_OP_M6',
 'SALDO_DEUDA_OP_M7',
 'SALDO_DEU

## 18. Persistencia

In [89]:
info.to_pickle("info.pkl")




In [90]:
info.head()

IDENTIFICACION TIPO_ID FECHA_CORTE ESTADO_CIVIL  CARGAS  \
0  244F70904DD738B089FC39D4432D2AB6       C  2022-09-30  Union Libre       1   
1  3B9DBDEE3B206ED41B865377DC2C48CE       C  2022-06-30  Soltero (a)       1   
2  C7D5591188D15C57CF396833AACADA88       C  2022-03-31  Soltero (a)       1   
3  491F4236D627EA9A823C9F065FDDA3CB       C  2022-03-31  Soltero (a)       0   
4  50207615DEE7456136F720BCE282EC79       C  2022-06-30  Soltero (a)       0   

  INSTRUCCION  ANTIG_LABORAL  ANTIG_DOMICILIARIA RELACION_DEPENDENCIA  \
0  Secundario             39                  15          DEPENDIENTE   
1  Secundario             29                   0          DEPENDIENTE   
2  Secundario             18                   0          DEPENDIENTE   
3  Secundario             36                   0         MICROEMPRESA   
4  Secundario             42                   0          DEPENDIENTE   

    PERFIL_CLIENTE  INGRESOS  GASTOS    PROVINCIA       CANTON    PARROQUIA  \
0      DEPENDIENTE    439.01  105.36  SANTA ELENA  LA LIBERTAD  LA LIBERTAD   
1      DEPENDIENTE    429.00  291.91          NaN          NaN          NaN   
2      DEPENDIENTE    471.96  136.34  SANTA ELENA  LA LIBERTAD  LA LIBERTAD   
3  MICROEMPRESARIO    295.50  163.27  SANTA ELENA  SANTA ELENA  SANTA ELENA   
4      DEPENDIENTE    661.89  316.75  SANTA ELENA  SANTA ELENA     COLONCHE   

  ACTIVIDAD_ECONOMICA REFINANCIAMIENTO_REESTRUCTURA MARCA_AYUDA  DPF COMUNAL  \
0             N000000                           NaN         NaN  NaN     NaN   
1             N000000                           NaN         NaN  NaN     NaN   
2             N000000                           NaN         NaN  NaN     NaN   
3             G471101                           NaN         NaN  NaN     NaN   
4             N000000                           NaN         NaN  NaN     NaN   

   MESES_GRACIA RENOVACION      OFICINA  SALDO_PROMEDIO_AHORRO  Muestra  \
0             0        NaN  LA LIBERTAD                   4.46        1   
1             0        NaN  LA LIBERTAD                  38.21        1   
2             0        NaN  LA LIBERTAD                   0.00        1   
3             0        NaN  LA LIBERTAD                   1.43        1   
4             0        NaN  LA LIBERTAD                2060.00        1   

   NUMERO_DIAS_MOROSIDAD_OP_M1  NUMERO_DIAS_MOROSIDAD_OP_M2  \
0                           22                           53   
1                            0                            0   
2                            0                            0   
3                            0                            0   
4                            0                            0   

   NUMERO_DIAS_MOROSIDAD_OP_M3  NUMERO_DIAS_MOROSIDAD_OP_M4  \
0                           51                            0   
1                            0                            0   
2                            0                            0   
3                            0                            0   
4                            0                            0   

   NUMERO_DIAS_MOROSIDAD_OP_M5  NUMERO_DIAS_MOROSIDAD_OP_M6  \
0                            0                           27   
1                            0                            0   
2                            0                            0   
3                            0                            0   
4                            0                            0   

   NUMERO_DIAS_MOROSIDAD_OP_M7  NUMERO_DIAS_MOROSIDAD_OP_M8  \
0                            0                            0   
1                            0                            1   
2                            0                            0   
3                            8                           30   
4                            0                            0   

   NUMERO_DIAS_MOROSIDAD_OP_M9  NUMERO_DIAS_MOROSIDAD_OP_M10  \
0                            0                            29   
1                            0                    

In [98]:
info.shape

(79591, 2565)